# 📚 03_统计分析与展示

> **核心能力**：分组统计 (groupby)、排序排名、文本提取、时间切片、NumPy 标签
>
> **开局心法**：`groupby` 是 SQL GROUP BY 的镜像；`transform` 能把结果广播回原表；`.str` 和 `.dt` 是文本和时间的瑞士军刀；`np.where/select` 是打标签的终极武器。

---

## 1️⃣ 分组统计：GroupBy（SQL GROUP BY 的镜像）

> **大白话口诀**：这是纯正的 SQL `GROUP BY` 的镜像。先分组，再在每组里求和/平均。

In [ ]:
import pandas as pd
import numpy as np

# 【SQL 视角】
# SELECT 季度, SUM(销量) AS 汇总销量 
# FROM df 
# GROUP BY 季度 
# ORDER BY 汇总销量 ASC;

# 【Pandas 镜像翻译】
result = df.groupby('季度')['销量'].sum().sort_values(ascending=True)

# 或者更详细的写法
result = df.groupby('季度').agg({'销量': 'sum'}).sort_values(by='销量', ascending=True)

In [ ]:
# 【更多 groupby 例子】

# 1. 🔢 [多列分组 + 多个统计]：按"品牌"和"季度"分组，同时求均价和总销量
summary = df.groupby(['品牌', '季度']).agg({
    '价格': 'mean',        # 均价
    '销量': 'sum'         # 总销量
})

# 2. 📊 [自定义聚合函数]：比如求最大值和最小值的差
df.groupby('品牌')['价格'].agg(['min', 'max', lambda x: x.max() - x.min()])

# 3. 🚀 [行转列展示]：reset_index() 把索引变回普通列，方便后续操作
summary_reset = summary.reset_index()

### 🚨 坑点：groupby 结果的索引陷阱

#### 坑点 1：groupby 后的结果被当成了索引，而不是普通列

```python
# ❌ 常见错误
result = df.groupby('品牌')['销量'].sum()
# 此时 result 的"品牌"不是列，而是黑粗体的索引！
# 后续想用 result['品牌'] 会报错

# ✅ 解决方案：reset_index()
result = df.groupby('品牌')['销量'].sum().reset_index()
# 现在 result 有两列：'品牌' 和 '销量'（都是普通列）
```

---

#### 坑点 2：groupby 后无法直接和其他表 merge

```python
# ❌ 因为"品牌"是索引，merge 时出错
grouped = df.groupby('品牌')['销量'].sum()
result = pd.merge(grouped, other_df, on='品牌')  # 报错！

# ✅ 先 reset_index
grouped = df.groupby('品牌')['销量'].sum().reset_index()
result = pd.merge(grouped, other_df, on='品牌')  # ✓
```

---
## 2️⃣ 排序与排名

### 【快速操作】找销售额最高的 Top 3 品牌

```python
top_3 = df.groupby('品牌')['销售额'].sum().sort_values(ascending=False).head(3)
```

In [ ]:
# 1. 📊 [按某列排序]：找"销售额"最高的前 10 行
top_10 = df.sort_values('销售额', ascending=False).head(10)

# 2. 🏅 [分组内排名]：在每个"品牌"内，按"价格"排名，看谁最贵
df['组内排名'] = df.groupby('品牌')['价格'].rank(method='dense', ascending=False)
# method='dense' 表示：1,1,3（跳过 2），不是 1,1,2
# method='first' 表示：1,2,3（有重复就给不同排名，按出现顺序）

# 3. 🚀 [找各品牌最贵的那一款]：
most_expensive = df.loc[df.groupby('品牌')['价格'].idxmax()]
# idxmax() 返回每组里最大值所在的索引位置

---
## 3️⃣ 窗口函数与数据广播（transform）

> **大白话口诀**：
> 不要把原表"折叠/弄短"，而是在最后面**加一列全局或者组内统计值**，好让当前行直接跟全局去比（比如：我比平均高多少？）。

这对应 SQL 的 `PARTITION BY` 窗口函数。

In [ ]:
# 1. 🌍 [全表统计广播]：每一行都显示"全市场总销售额"
df['全市场销售额'] = df['销售额'].sum()
df['市场占比'] = (df['销售额'] / df['全市场销售额'] * 100).round(2)

# 2. 👥 [分组统计广播]：每行显示"自己所在品牌的平均价格"
# transform 的魔力就在这里：返回的结果长度和原表一样！
df['本品牌均价'] = df.groupby('品牌')['价格'].transform('mean')
df['我比品牌均价高多少'] = df['价格'] - df['本品牌均价']

# 3. 🎖️ [分组排名]：在每个品牌内排名，保留在原表中
df['品牌内排名'] = df.groupby('品牌')['销售额'].transform(lambda x: x.rank(method='dense', ascending=False))

# 4. 💰 [累计求和]：按时间顺序，从第 1 行开始累加销售额
df = df.sort_values('日期').reset_index(drop=True)
df['累计销售额'] = df['销售额'].cumsum()

### 【SQL 对照】窗口函数

```sql
-- SQL 版本
SELECT 
    *,
    SUM(销售额) OVER() AS 全市场销售额,
    AVG(价格) OVER(PARTITION BY 品牌) AS 本品牌均价,
    DENSE_RANK() OVER(PARTITION BY 品牌 ORDER BY 销售额 DESC) AS 品牌内排名,
    SUM(销售额) OVER(ORDER BY 日期) AS 累计销售额
FROM df;
```

**Pandas 对应**：`transform` 就是你的「窗口函数」

---
## 4️⃣ 文本处理：`.str` 大法

> **大白话口诀**：不要用 `for` 循环和 `apply` 去一行行查字！用 Pandas 自带的 `.str` 文本方法，速度快 100 倍。这对应 SQL 里的 `LIKE '%关键词%'`。

In [ ]:
# 1. 🔍 [检查包含]：判断"显卡名"是否包含"RTX"或"GDDR"
keywords = 'RTX|GDDR'
df['含新显卡'] = df['显卡名'].str.upper().str.contains(keywords, na=False).astype(int)

# 2. ✂️ [提取子串]：从"北京市朝阳区"提取前 2 个字（"北京"）
df['城市代码'] = df['地址'].str[:2]  # 切前 2 个字

# 3. 🔗 [合并字符串]：把"品牌"和"型号"拼成"完整名"
df['完整名'] = df['品牌'] + '-' + df['型号']

# 4. 🧹 [清理空格和特殊符号]：见前面的"脏字符清理"章节
df['干净名'] = df['原始名'].str.replace(r'[\s,]', '', regex=True).str.strip()

---
## 5️⃣ 时间处理：`.dt` 大法

> **大白话口诀**：老板要按"月"看报表，但你的数据是 `"2026-03-23 15:30:00"` 这种精确到秒的怎么办？
> 不要用 `str` 切割！先用 `pd.to_datetime` 把它变成 Pandas 认识的时间老人，然后用 `.dt.year / .dt.month` 瞬间抽取出你想要的部分来做 `groupby`！

In [ ]:
# 1. 🧙‍♂️ [点石成金：字符串变时间格式]
# 第一步永远是把 "2026-03-23" 这种文本字符串，转化为被 Pandas 认可的时间类型
df['下单时间'] = pd.to_datetime(df['下单时间'])

# 2. ✂️ [切片大师：用 .dt 拿取任意时间颗粒度]
# 前提：你的列已经是 datetime 格式了，然后加上 .dt 就能随便拿
df['年份'] = df['下单时间'].dt.year            # 提取：2026
df['月份'] = df['下单时间'].dt.month           # 提取：3
df['几号'] = df['下单时间'].dt.day             # 提取：23
df['星期几'] = df['下单时间'].dt.dayofweek      # 提取：0 (周一) ~ 6 (周日)
df['只拿日期'] = df['下单时间'].dt.date          # 砍掉时分秒

# 3. 📊 [实战连招]：老板要看每个月的总销售额
df['交易月'] = pd.to_datetime(df['购买时间']).dt.to_period('M')  # 转成"202603"这种格式
monthly_sales = df.groupby('交易月')['金额'].sum()

### 🚨 坑点：datetime 转换的坑

#### 坑点 1：pd.to_datetime 失败，报错"unknown datetime format"

```python
# ❌ 格式混乱（有的"2026-03-23"，有的"2026/3/23"，有的"23-03-2026"）
df['日期'] = pd.to_datetime(df['日期'])  # 报错或结果混乱

# ✅ 指定格式或用 infer_datetime_format
df['日期'] = pd.to_datetime(df['日期'], format='%Y-%m-%d')
# 或
df['日期'] = pd.to_datetime(df['日期'], infer_datetime_format=True)
```

---

#### 坑点 2：dayofweek 的编号容易搞反

```python
# 💡 记住这个：0=周一, 1=周二, ..., 6=周日
# 如果你想要周一=1, 周日=7 的 SQL 风格
df['sql_day_of_week'] = df['日期'].dt.dayofweek + 1
```

---
## 6️⃣ 打标签的终极武器：`np.where` & `np.select`

> **大白话口诀**：当 Pandas 的条件过于复杂，或者需要多档位打标签，用 NumPy 秒杀。这对应 SQL 的 `CASE WHEN`。

In [ ]:
# 1. 🚦 [两档开关：np.where]
# np.where(条件, 是的话填什么, 不是的话填什么)
df['定位'] = np.where(df['价格'] > 8000, '高端', '普通')

# 2. 🚦 [多档开关：np.select] (SQL CASE WHEN 的超级版本)
conditions = [
    (df['价格'] >= 10000), 
    (df['价格'] >= 6000) & (df['价格'] < 10000), 
    (df['价格'] < 6000)
]
choices = ['顶级尊享', '主流配置', '垃圾入门']
df['细分市场'] = np.select(conditions, choices, default='其他未知')

# 【SQL 对比】
# CASE WHEN 价格 >= 10000 THEN '顶级尊享'
#      WHEN 价格 >= 6000 THEN '主流配置'
#      WHEN 价格 < 6000 THEN '垃圾入门'
#      ELSE '其他未知'
# END

### 💡 np.where vs pd.cut vs np.select 的选择

| 方法 | 场景 | 速度 | 易用性 |
|------|------|------|--------|
| `np.where` | 只需要两档分类 | ⭐⭐⭐⭐⭐ 最快 | ⭐⭐⭐⭐⭐ 最简单 |
| `np.select` | 需要 3+ 档分类，条件复杂 | ⭐⭐⭐⭐ | ⭐⭐⭐⭐ |
| `pd.cut` | 连续数值分档，比如按"价格区间"自动分 5 档 | ⭐⭐⭐ | ⭐⭐⭐ |
| `pd.loc + mask` | 多次赋值（很不推荐） | ⭐ 最慢 | ⭐ 最麻烦 |

---
## 【连招】常用数据分析链路

### 场景：我要分析"各品牌在不同月份的销售情况"

```python
# 第 1 步：准备数据
df['交易月'] = pd.to_datetime(df['日期']).dt.to_period('M')

# 第 2 步：分组统计
summary = df.groupby(['品牌', '交易月']).agg({
    '销售额': 'sum',
    '销售量': 'mean'
}).reset_index()

# 第 3 步：广播计算（我比平均高多少）
summary['品牌月均'] = summary.groupby('品牌')['销售额'].transform('mean')
summary['对比'] = summary['销售额'] - summary['品牌月均']

# 第 4 步：打标签（是否超过平均）
summary['达标'] = np.where(summary['对比'] > 0, '✓ 超额', '✗ 未达')

# 第 5 步：排序查看
result = summary.sort_values('销售额', ascending=False).head(10)
```